In [1]:

import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("UrbanRent_Analytics")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.sql.crossJoin.enabled", "true")
  .getOrCreate()

import spark.implicits._
spark.sparkContext.setLogLevel("ERROR")

println(s"✅ UrbanRent Analytics iniciado — Spark ${spark.version} · Scala ${scala.util.Properties.versionString}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/30 12:29:49 INFO SparkContext: Running Spark version 4.1.1
26/04/30 12:29:49 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/30 12:29:49 INFO SparkContext: Java version 17.0.12+8-LTS-286
26/04/30 12:29:49 INFO ResourceUtils: ==============================================================
26/04/30 12:29:49 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/30 12:29:49 INFO ResourceUtils: ==============================================================
26/04/30 12:29:49 INFO SparkContext: Submitted application: UrbanRent_Analytics
26/04/30 12:29:49 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/30 12:29:49 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/30 12:29:49 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/30 12:29:49 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/30 12:29:49 INFO SecurityManager: Changing vie

2026-04-30T10:29:49.669524Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.cor

import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.types._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@49a9c3f9
import spark.implicits._

In [8]:
import java.io._

val contenidoCSV = """id_reserva,id_apartamento,fecha_entrada,fecha_salida,num_huespedes,precio_noche,canal,valoracion,ciudad
R001,APT001,2025-01-10,2025-01-13,2,95.0,Airbnb,4.8,Madrid
R002,APT002,2025-01-12,2025-01-15,4,120.0,Booking,4.5,Barcelona
R003,APT003,2025-01-20,2025-01-22,1,75.0,Directo,5.0,Valencia
R004,APT001,2025-02-01,2025-02-05,3,95.0,Airbnb,4.7,Madrid
R005,APT004,2025-02-10,2025-02-12,2,110.0,Booking,4.2,Sevilla
R006,APT002,2025-02-14,2025-02-17,5,120.0,Airbnb,4.9,Barcelona
R007,APT005,2025-02-20,2025-02-23,2,85.0,Directo,4.6,Bilbao
R008,APT003,2025-03-01,2025-03-04,1,75.0,Booking,4.4,Valencia
R009,APT001,2025-03-10,2025-03-14,4,95.0,Airbnb,4.8,Madrid
R010,APT006,2025-03-15,2025-03-17,2,130.0,Directo,5.0,Madrid
R011,APT004,2025-03-20,2025-03-22,3,110.0,Airbnb,4.3,Sevilla
R012,APT007,2025-04-01,2025-04-04,2,90.0,Booking,4.5,Barcelona
R013,APT005,2025-04-08,2025-04-10,1,85.0,Directo,4.7,Bilbao
R014,APT006,2025-04-12,2025-04-16,3,130.0,Airbnb,4.9,Madrid
R015,APT002,2025-04-20,2025-04-23,4,120.0,Booking,4.6,Barcelona
R016,APT008,2025-05-01,2025-05-04,2,70.0,Directo,4.1,Valencia
R017,APT003,2025-05-10,2025-05-13,2,75.0,Airbnb,4.5,Valencia
R018,APT007,2025-05-15,2025-05-18,3,90.0,Booking,4.8,Barcelona
R019,APT001,2025-05-20,2025-05-24,2,95.0,Directo,5.0,Madrid
R020,APT009,2025-05-25,2025-05-28,1,65.0,Airbnb,3.9,Bilbao
R021,APT006,2025-06-01,2025-06-05,4,130.0,Booking,4.7,Madrid
R022,APT010,2025-06-08,2025-06-10,2,80.0,Directo,4.3,Sevilla
R023,APT004,2025-06-15,2025-06-18,3,110.0,Airbnb,4.6,Sevilla
R024,APT008,2025-06-20,2025-06-22,1,70.0,Booking,4.2,Valencia
R025,APT005,2025-06-25,2025-06-28,2,85.0,Directo,4.8,Bilbao"""

val file = new File("reservas.csv")
val bw = new BufferedWriter(new FileWriter(file))
bw.write(contenidoCSV)
bw.close()

println("Archivo creado con éxito en: " + file.getAbsolutePath)


Archivo creado con éxito en: c:\Users\Imp_06 - Mañana\Desktop\Spark\reservas.csv


import java.io._
contenidoCSV: String = """id_reserva,id_apartamento,fecha_entrada,fecha_salida,num_huespedes,precio_noche,canal,valoracion,ciudad
R001,APT001,2025-01-10,2025-01-13,2,95.0,Airbnb,4.8,Madrid
R002,APT002,2025-01-12,2025-01-15,4,120.0,Booking,4.5,Barcelona
R003,APT003,2025-01-20,2025-01-22,1,75.0,Directo,5.0,Valencia
R004,APT001,2025-02-01,2025-02-05,3,95.0,Airbnb,4.7,Madrid
R005,APT004,2025-02-10,2025-02-12,2,110.0,Booking,4.2,Sevilla
R006,APT002,2025-02-14,2025-02-17,5,120.0,Airbnb,4.9,Barcelona
R007,APT005,2025-02-20,2025-02-23,2,85.0,Directo,4.6,Bilbao
R008,APT003,2025-03-01,2025-03-04,1,75.0,Booking,4.4,Valencia
R009,APT001,2025-03-10,2025-03-14,4,95.0,Airbnb,4.8,Madrid
R010,APT006,2025-03-15,2025-03-17,2,130.0,Directo,5.0,Madrid
R011,APT004,2025-03-20,2025-03-22,3,110.0,Airbnb,4.3,Sevilla
R012,APT007,2025-04-01,2025-04-04,2,90.0,Booking,4.5,Barcelona
R013,APT005,2025-04-08,2025-04-10,1,85.0,Directo,4.7,Bilbao
R014,APT006,2025-04-12,2025-04-16,3,130.0,Airbnb,4.9,Madri

Pregunta 1 — ¿Están bien formateados nuestros datos de reservas?
El responsable de datos quiere verificar que Spark ha inferido correctamente los tipos antes de avanzar con cualquier análisis.

Lo que debes hacer:

Carga reservas.csv con header = true e inferSchema = true. Llama al DataFrame reservas.
Muestra las primeras 8 filas con show(8).
Imprime el schema completo con printSchema().
Muestra estadísticas descriptivas de precio_noche, num_huespedes y valoracion con describe().
Imprime el número total de reservas con count() y lista todas las columnas con columns.

In [10]:
val reservas = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("reservas.csv")

reservas.show(8)
reservas.printSchema()

reservas.select("precio_noche", "num_huespedes", "valoracion").describe().show()

val totalReservas = reservas.count()
println(s"Número total de reservas: $totalReservas")

// Solo declara esto una vez
val nombresColumnas = reservas.columns
println("Columnas en el DataFrame: " + nombresColumnas.mkString(", "))


+----------+--------------+-------------+------------+-------------+------------+-------+----------+---------+
|id_reserva|id_apartamento|fecha_entrada|fecha_salida|num_huespedes|precio_noche|  canal|valoracion|   ciudad|
+----------+--------------+-------------+------------+-------------+------------+-------+----------+---------+
|      R001|        APT001|   2025-01-10|  2025-01-13|            2|        95.0| Airbnb|       4.8|   Madrid|
|      R002|        APT002|   2025-01-12|  2025-01-15|            4|       120.0|Booking|       4.5|Barcelona|
|      R003|        APT003|   2025-01-20|  2025-01-22|            1|        75.0|Directo|       5.0| Valencia|
|      R004|        APT001|   2025-02-01|  2025-02-05|            3|        95.0| Airbnb|       4.7|   Madrid|
|      R005|        APT004|   2025-02-10|  2025-02-12|            2|       110.0|Booking|       4.2|  Sevilla|
|      R006|        APT002|   2025-02-14|  2025-02-17|            5|       120.0| Airbnb|       4.9|Barcelona|
|

reservas: org.apache.spark.sql.package.DataFrame = [id_reserva: string, id_apartamento: string ... 7 more fields]
totalReservas: Long = 25L
nombresColumnas: Array[String] = Array(
  "id_reserva",
  "id_apartamento",
  "fecha_entrada",
  "fecha_salida",
  "num_huespedes",
  "precio_noche",
  "canal",
  "valoracion",
  "ciudad"
)

Pregunta 2 — ¿Es correcto el schema inferido o necesitamos ajustarlo?
El equipo técnico señala que fecha_entrada y fecha_salida llegan como StringType. Para los cálculos de duración de estancias necesitarán ser DateType.

Lo que debes hacer:

Define un StructType manual para reservas con al menos estos ajustes:
fecha_entrada y fecha_salida como DateType (usa to_date o define el tipo en el schema).
precio_noche como DoubleType.
valoracion como DoubleType.
num_huespedes como IntegerType.
Recarga reservas.csv usando .schema(...) en lugar de inferSchema.
Verifica con printSchema() que los tipos son ahora los correctos.
Pista: Para leer fechas con schema manual en CSV usa .option("dateFormat", "yyyy-MM-dd").

In [ ]:
import org.apache.spark.sql.types._

val manualSchema = StructType(Array(
  StructField("id_reserva", StringType, true),
  StructField("id_apartamento", StringType, true),
  StructField("fecha_entrada", DateType, true),  // Ajustado a DateType
  StructField("fecha_salida", DateType, true),   // Ajustado a DateType
  StructField("num_huespedes", IntegerType, true), // Ajustado a IntegerType
  StructField("precio_noche", DoubleType, true),   // Ajustado a DoubleType
  StructField("canal", StringType, true),
  StructField("valoracion", DoubleType, true),     // Ajustado a DoubleType
  StructField("ciudad", StringType, true)
))


val reservas = spark.read
  .option("header", "true")
  .option("dateFormat", "yyyy-MM-dd") 
  .schema(manualSchema)               
  .csv("reservas.csv")


reservas.printSchema()
reservas.show(5)


root
 |-- id_reserva: string (nullable = true)
 |-- id_apartamento: string (nullable = true)
 |-- fecha_entrada: date (nullable = true)
 |-- fecha_salida: date (nullable = true)
 |-- num_huespedes: integer (nullable = true)
 |-- precio_noche: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- valoracion: double (nullable = true)
 |-- ciudad: string (nullable = true)

+----------+--------------+-------------+------------+-------------+------------+-------+----------+---------+
|id_reserva|id_apartamento|fecha_entrada|fecha_salida|num_huespedes|precio_noche|  canal|valoracion|   ciudad|
+----------+--------------+-------------+------------+-------------+------------+-------+----------+---------+
|      R001|        APT001|   2025-01-10|  2025-01-13|            2|        95.0| Airbnb|       4.8|   Madrid|
|      R002|        APT002|   2025-01-12|  2025-01-15|            4|       120.0|Booking|       4.5|Barcelona|
|      R003|        APT003|   2025-01-20|  2025-01-22|     

import org.apache.spark.sql.types._
manualSchema: StructType = Seq(
  StructField(
    name = "id_reserva",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "id_apartamento",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "fecha_entrada",
    dataType = DateType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "fecha_salida",
    dataType = DateType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "num_huespedes",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "precio_noche",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "canal",
...
reservas: org.apache.spark.sql.package.DataFrame = [id_reserva: string, id_apartamento: string ... 7 more fields]

Pregunta 3 — ¿Cuántos apartamentos tenemos y cuáles son sus características?
Lo que debes hacer:

Carga apartamentos.csv con inferSchema = true. Llama al DataFrame apartamentos.
Carga propietarios.json con multiline = true. Llama al DataFrame propietarios.
Para cada uno: muestra el schema, las primeras filas y el count().
¿Cuántos apartamentos están marcados como activo = true?


In [12]:
import java.io._

val datosApartamentos = """id_apartamento,tipo,habitaciones,metros_cuadrados,tiene_parking,propietario_id,activo
APT001,Estudio,1,35,false,P01,true
APT002,Piso,3,85,true,P02,true
APT003,Estudio,1,30,false,P01,true
APT004,Piso,2,65,true,P03,true
APT005,Ático,2,70,true,P04,true
APT006,Piso,3,90,true,P02,true
APT007,Estudio,1,28,false,P05,true
APT008,Estudio,1,32,false,P03,false
APT009,Piso,2,55,false,P04,true
APT010,Ático,3,100,true,P05,true"""


val fileApt = new File("apartamentos.csv")
val bwApt = new BufferedWriter(new FileWriter(fileApt))
bwApt.write(datosApartamentos)
bwApt.close()


val apartamentos = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("apartamentos.csv")


apartamentos.printSchema()
apartamentos.show(5)
println(s"Total apartamentos: ${apartamentos.count()}")


val activosCount = apartamentos.filter("activo = true").count()
println(s"\nApartamentos activos: $activosCount")


root
 |-- id_apartamento: string (nullable = true)
 |-- tipo: string (nullable = true)
 |-- habitaciones: integer (nullable = true)
 |-- metros_cuadrados: integer (nullable = true)
 |-- tiene_parking: boolean (nullable = true)
 |-- propietario_id: string (nullable = true)
 |-- activo: boolean (nullable = true)

+--------------+-------+------------+----------------+-------------+--------------+------+
|id_apartamento|   tipo|habitaciones|metros_cuadrados|tiene_parking|propietario_id|activo|
+--------------+-------+------------+----------------+-------------+--------------+------+
|        APT001|Estudio|           1|              35|        false|           P01|  true|
|        APT002|   Piso|           3|              85|         true|           P02|  true|
|        APT003|Estudio|           1|              30|        false|           P01|  true|
|        APT004|   Piso|           2|              65|         true|           P03|  true|
|        APT005|  �tico|           2|             

import java.io._
datosApartamentos: String = """id_apartamento,tipo,habitaciones,metros_cuadrados,tiene_parking,propietario_id,activo
APT001,Estudio,1,35,false,P01,true
APT002,Piso,3,85,true,P02,true
APT003,Estudio,1,30,false,P01,true
APT004,Piso,2,65,true,P03,true
APT005,Ático,2,70,true,P04,true
APT006,Piso,3,90,true,P02,true
APT007,Estudio,1,28,false,P05,true
APT008,Estudio,1,32,false,P03,false
APT009,Piso,2,55,false,P04,true
APT010,Ático,3,100,true,P05,true"""
fileApt: File = apartamentos.csv
bwApt: BufferedWriter = java.io.BufferedWriter@53c39943
apartamentos: org.apache.spark.sql.package.DataFrame = [id_apartamento: string, tipo: string ... 5 more fields]
activosCount: Long = 9L

Pregunta 5 — ¿Qué canales de venta tenemos activos y cuál es el precio medio por canal?
El equipo de marketing quiere un catálogo limpio de canales sin repetidos.

Lo que debes hacer:

Sobre reservas, elimina duplicados por la columna canal.
Selecciona únicamente las columnas canal y precio_noche.
Ordena de mayor a menor precio_noche.
Pista: Usa dropDuplicates("canal").

In [17]:
import org.apache.spark.sql.functions.col

val canalesUnicos = reservas
  .dropDuplicates("canal")
  .select("canal", "precio_noche")
  .orderBy(col("precio_noche").desc)

canalesUnicos.show()


+-------+------------+
|  canal|precio_noche|
+-------+------------+
|Directo|       130.0|
|Booking|       120.0|
| Airbnb|        95.0|
+-------+------------+



import org.apache.spark.sql.functions.col
canalesUnicos: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [canal: string, precio_noche: double]

Pregunta 6 — ¿Cuánto ingresaría cada reserva si aplicáramos un suplemento del 10% para reservas con más de 2 huéspedes?
Lo que debes hacer:

Añade una columna precio_con_suplemento usando when/otherwise:
Si num_huespedes > 2: precio_noche * 1.10
En otro caso: precio_noche
Añade ingreso_estimado = num_noches * precio_con_suplemento (asegúrate de que num_noches existe en el DataFrame).
Selecciona id_reserva, ciudad, num_huespedes, precio_noche, precio_con_suplemento, ingreso_estimado.
Ordena por ingreso_estimado descendente.


In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.{datediff, to_date, col, when}

val spark = SparkSession.builder()
  .appName("Pregunta6")
  .master("local[*]")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

val datosReservas = Seq(
  ("R001","2025-01-10","2025-01-13",95.0,2,"Madrid"),
  ("R002","2025-01-12","2025-01-15",120.0,4,"Barcelona"),
  ("R003","2025-01-20","2025-01-22",75.0,1,"Valencia"),
  ("R004","2025-02-01","2025-02-05",95.0,3,"Madrid"),
  ("R005","2025-02-10","2025-02-12",110.0,2,"Sevilla")
)

val schema = StructType(Array(
  StructField("id_reserva", StringType, true),
  StructField("fecha_entrada", StringType, true),
  StructField("fecha_salida", StringType, true),
  StructField("precio_noche", DoubleType, true),
  StructField("num_huespedes", IntegerType, true),
  StructField("ciudad", StringType, true)
))

val rdd = spark.sparkContext.parallelize(datosReservas).map(p => Row(p._1, p._2, p._3, p._4, p._5, p._6))
val reservas = spark.createDataFrame(rdd, schema)

val resultado6 = reservas
  .withColumn("num_noches", datediff(to_date(col("fecha_salida"), "yyyy-MM-dd"), to_date(col("fecha_entrada"), "yyyy-MM-dd")))
  // .otherwise() se ejecuta encadenado a .when() de forma nativa
  .withColumn("precio_con_suplemento", when(col("num_huespedes") > 2, col("precio_noche") * 1.10).otherwise(col("precio_noche")))
  .withColumn("ingreso_estimado", col("num_noches") * col("precio_con_suplemento"))
  .select("id_reserva", "ciudad", "num_huespedes", "precio_noche", "precio_con_suplemento", "ingreso_estimado")
  .orderBy(col("ingreso_estimado").desc)

resultado6.show()


+----------+---------+-------------+------------+---------------------+------------------+
|id_reserva|   ciudad|num_huespedes|precio_noche|precio_con_suplemento|  ingreso_estimado|
+----------+---------+-------------+------------+---------------------+------------------+
|      R004|   Madrid|            3|        95.0|   104.50000000000001|418.00000000000006|
|      R002|Barcelona|            4|       120.0|                132.0|             396.0|
|      R001|   Madrid|            2|        95.0|                 95.0|             285.0|
|      R005|  Sevilla|            2|       110.0|                110.0|             220.0|
|      R003| Valencia|            1|        75.0|                 75.0|             150.0|
+----------+---------+-------------+------------+---------------------+------------------+



import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.{datediff, to_date, col, when}
spark: SparkSession = org.apache.spark.sql.SparkSession@3a466b35
datosReservas: Seq[(String, String, String, Double, Int, String)] = List(
  ("R001", "2025-01-10", "2025-01-13", 95.0, 2, "Madrid"),
  ("R002", "2025-01-12", "2025-01-15", 120.0, 4, "Barcelona"),
  ("R003", "2025-01-20", "2025-01-22", 75.0, 1, "Valencia"),
  ("R004", "2025-02-01", "2025-02-05", 95.0, 3, "Madrid"),
  ("R005", "2025-02-10", "2025-02-12", 110.0, 2, "Sevilla")
)
schema: StructType = Seq(
  StructField(
    name = "id_reserva",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "fecha_entrada",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "fecha_salida",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(


Pregunta 7 — ¿Existen ciudades con reservas en el dataset de apartamentos que no aparecen en el de reservas?
Lo que debes hacer:

Renombra la columna ciudad del DataFrame de propietarios a ciudad_propietario para evitar ambigüedades.
Muestra las ciudades únicas en reservas con select("ciudad").distinct().
Muestra las ciudades únicas en propietarios (columna ciudad) con select("ciudad").distinct().

In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.{datediff, to_date, col, sum, count}

val spark = SparkSession.builder()
  .appName("DominioCanales")
  .master("local[*]")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

val datosReservas = Seq(
  ("R001","2025-01-10","2025-01-13",95.0,"Airbnb","Madrid"),
  ("R002","2025-01-12","2025-01-15",120.0,"Booking","Barcelona"),
  ("R003","2025-01-20","2025-01-22",75.0,"Directo","Valencia"),
  ("R004","2025-02-01","2025-02-05",95.0,"Airbnb","Madrid"),
  ("R005","2025-02-10","2025-02-12",110.0,"Booking","Sevilla"),
  ("R007","2025-02-20","2025-02-23",85.0,"Directo","Bilbao"),
  ("R008","2025-03-01","2025-03-04",75.0,"Booking","Valencia"),
  ("R013","2025-04-08","2025-04-10",85.0,"Directo","Bilbao"),
  ("R016","2025-05-01","2025-05-04",70.0,"Directo","Valencia"),
  ("R020","2025-05-25","2025-05-28",65.0,"Airbnb","Bilbao")
)

val schema = StructType(Array(
  StructField("id_reserva", StringType, true),
  StructField("fecha_entrada", StringType, true),
  StructField("fecha_salida", StringType, true),
  StructField("precio_noche", DoubleType, true),
  StructField("canal", StringType, true),
  StructField("ciudad", StringType, true)
))

val rdd = spark.sparkContext.parallelize(datosReservas).map(p => Row(p._1, p._2, p._3, p._4, p._5, p._6))
val reservas = spark.createDataFrame(rdd, schema)


val dfAnalisisCanal = reservas
  .withColumn("num_noches", datediff(to_date(col("fecha_salida"), "yyyy-MM-dd"), to_date(col("fecha_entrada"), "yyyy-MM-dd")))
  .withColumn("ingreso_reserva", col("num_noches") * col("precio_noche"))

val dominioCanalPorCiudad = dfAnalisisCanal
  .groupBy("ciudad", "canal")
  .agg(
    sum("ingreso_reserva").alias("ingreso_total"),
    count("id_reserva").alias("num_reservas")
  )
  .orderBy(col("ciudad").asc, col("ingreso_total").desc)

println("--- DOMINIO DE CANALES POR CIUDAD ---")
dominioCanalPorCiudad.show()


--- DOMINIO DE CANALES POR CIUDAD ---
+---------+-------+-------------+------------+
|   ciudad|  canal|ingreso_total|num_reservas|
+---------+-------+-------------+------------+
|Barcelona|Booking|        360.0|           1|
|   Bilbao|Directo|        425.0|           2|
|   Bilbao| Airbnb|        195.0|           1|
|   Madrid| Airbnb|        665.0|           2|
|  Sevilla|Booking|        220.0|           1|
| Valencia|Directo|        360.0|           2|
| Valencia|Booking|        225.0|           1|
+---------+-------+-------------+------------+



import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.{datediff, to_date, col, sum, count}
spark: SparkSession = org.apache.spark.sql.SparkSession@3a466b35
datosReservas: Seq[(String, String, String, Double, String, String)] = List(
  ("R001", "2025-01-10", "2025-01-13", 95.0, "Airbnb", "Madrid"),
  ("R002", "2025-01-12", "2025-01-15", 120.0, "Booking", "Barcelona"),
  ("R003", "2025-01-20", "2025-01-22", 75.0, "Directo", "Valencia"),
  ("R004", "2025-02-01", "2025-02-05", 95.0, "Airbnb", "Madrid"),
  ("R005", "2025-02-10", "2025-02-12", 110.0, "Booking", "Sevilla"),
  ("R007", "2025-02-20", "2025-02-23", 85.0, "Directo", "Bilbao"),
  ("R008", "2025-03-01", "2025-03-04", 75.0, "Booking", "Valencia"),
  ("R013", "2025-04-08", "2025-04-10", 85.0, "Directo", "Bilbao"),
  ("R016", "2025-05-01", "2025-05-04", 70.0, "Directo", "Valencia"),
  ("R020", "2025-05-25", "2025-05-28", 65.0, "Airbnb", "Bilba

Pregunta 8 — ¿Qué ciudad genera más ingresos para la empresa?
Lo que debes hacer:

Parte del DataFrame que ya tiene la columna ingreso_reserva (Pregunta 4, o recalcúlala si es necesario).
Agrupa por ciudad y calcula:
num_reservas = número de reservas
ingreso_total = suma de ingreso_reserva
ticket_medio = promedio de ingreso_reserva
valoracion_media = promedio de valoracion
Ordena por ingreso_total descendente.


In [8]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.{datediff, to_date, sum, count, avg, col}

val spark = SparkSession.builder()
  .appName("Pregunta8")
  .master("local[*]")
  .getOrCreate()

// ESTA LÍNEA APAGA LOS LOGS PARA QUE NO OCUPEN ESPACIO
spark.sparkContext.setLogLevel("ERROR")

val datosReservas = Seq(
  ("R001","2025-01-10","2025-01-13",95.0,4.8,"Madrid"),
  ("R002","2025-01-12","2025-01-15",120.0,4.5,"Barcelona"),
  ("R003","2025-01-20","2025-01-22",75.0,5.0,"Valencia"),
  ("R004","2025-02-01","2025-02-05",95.0,4.7,"Madrid"),
  ("R005","2025-02-10","2025-02-12",110.0,4.2,"Sevilla"),
  ("R007","2025-02-20","2025-02-23",85.0,4.6,"Bilbao"),
  ("R008","2025-03-01","2025-03-04",75.0,4.4,"Valencia"),
  ("R013","2025-04-08","2025-04-10",85.0,4.7,"Bilbao"),
  ("R016","2025-05-01","2025-05-04",70.0,4.1,"Valencia"),
  ("R020","2025-05-25","2025-05-28",65.0,3.9,"Bilbao")
)

val schema = StructType(Array(
  StructField("id_reserva", StringType, true),
  StructField("fecha_entrada", StringType, true),
  StructField("fecha_salida", StringType, true),
  StructField("precio_noche", DoubleType, true),
  StructField("valoracion", DoubleType, true),
  StructField("ciudad", StringType, true)
))

val rdd = spark.sparkContext.parallelize(datosReservas).map(p => Row(p._1, p._2, p._3, p._4, p._5, p._6))
val reservas = spark.createDataFrame(rdd, schema)

val reservasConIngreso = reservas
  .withColumn("num_noches", datediff(to_date(col("fecha_salida"), "yyyy-MM-dd"), to_date(col("fecha_entrada"), "yyyy-MM-dd")))
  .withColumn("ingreso_reserva", col("num_noches") * col("precio_noche"))

val metricasCiudad = reservasConIngreso
  .groupBy("ciudad")
  .agg(
    count("*").as("num_reservas"),
    sum("ingreso_reserva").as("ingreso_total"),
    avg("ingreso_reserva").as("ticket_medio"),
    avg("valoracion").as("valoracion_media")
  )
  .orderBy(col("ingreso_total").desc)

metricasCiudad.show()


+---------+------------+-------------+------------------+----------------+
|   ciudad|num_reservas|ingreso_total|      ticket_medio|valoracion_media|
+---------+------------+-------------+------------------+----------------+
|   Madrid|           2|        665.0|             332.5|            4.75|
|   Bilbao|           3|        620.0|206.66666666666666|             4.4|
| Valencia|           3|        585.0|             195.0|             4.5|
|Barcelona|           1|        360.0|             360.0|             4.5|
|  Sevilla|           1|        220.0|             220.0|             4.2|
+---------+------------+-------------+------------------+----------------+



import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.{datediff, to_date, sum, count, avg, col}
spark: SparkSession = org.apache.spark.sql.SparkSession@3a466b35
datosReservas: Seq[(String, String, String, Double, Double, String)] = List(
  ("R001", "2025-01-10", "2025-01-13", 95.0, 4.8, "Madrid"),
  ("R002", "2025-01-12", "2025-01-15", 120.0, 4.5, "Barcelona"),
  ("R003", "2025-01-20", "2025-01-22", 75.0, 5.0, "Valencia"),
  ("R004", "2025-02-01", "2025-02-05", 95.0, 4.7, "Madrid"),
  ("R005", "2025-02-10", "2025-02-12", 110.0, 4.2, "Sevilla"),
  ("R007", "2025-02-20", "2025-02-23", 85.0, 4.6, "Bilbao"),
  ("R008", "2025-03-01", "2025-03-04", 75.0, 4.4, "Valencia"),
  ("R013", "2025-04-08", "2025-04-10", 85.0, 4.7, "Bilbao"),
  ("R016", "2025-05-01", "2025-05-04", 70.0, 4.1, "Valencia"),
  ("R020", "2025-05-25", "2025-05-28", 65.0, 3.9, "Bilbao")
)
schema: StructType = Seq(
  StructField(
    n

Pregunta 9 — ¿Cuál es el canal de venta más rentable por ciudad?
La dirección quiere saber si Airbnb, Booking o el canal Directo domina en cada ciudad.

Lo que debes hacer:

Agrupa por ciudad y canal.
Calcula ingreso_total = suma de ingreso_reserva y num_reservas = count.
Ordena por ciudad y ingreso_total descendente.

In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.{datediff, to_date, col, sum, count}


val spark = SparkSession.builder()
  .appName("Pregunta9")
  .master("local[*]")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


val datosReservas = Seq(
  ("R001","2025-01-10","2025-01-13",95.0,"Airbnb","Madrid"),
  ("R002","2025-01-12","2025-01-15",120.0,"Booking","Barcelona"),
  ("R003","2025-01-20","2025-01-22",75.0,"Directo","Valencia"),
  ("R004","2025-02-01","2025-02-05",95.0,"Airbnb","Madrid"),
  ("R005","2025-02-10","2025-02-12",110.0,"Booking","Sevilla"),
  ("R007","2025-02-20","2025-02-23",85.0,"Directo","Bilbao"),
  ("R008","2025-03-01","2025-03-04",75.0,"Booking","Valencia"),
  ("R013","2025-04-08","2025-04-10",85.0,"Directo","Bilbao"),
  ("R016","2025-05-01","2025-05-04",70.0,"Directo","Valencia"),
  ("R020","2025-05-25","2025-05-28",65.0,"Airbnb","Bilbao")
)

val schema = StructType(Array(
  StructField("id_reserva", StringType, true),
  StructField("fecha_entrada", StringType, true),
  StructField("fecha_salida", StringType, true),
  StructField("precio_noche", DoubleType, true),
  StructField("canal", StringType, true),
  StructField("ciudad", StringType, true)
))

val rdd = spark.sparkContext.parallelize(datosReservas).map(p => Row(p._1, p._2, p._3, p._4, p._5, p._6))
val reservas = spark.createDataFrame(rdd, schema)

val dfConIngreso = reservas
  .withColumn("num_noches", datediff(to_date(col("fecha_salida"), "yyyy-MM-dd"), to_date(col("fecha_entrada"), "yyyy-MM-dd")))
  .withColumn("ingreso_reserva", col("num_noches") * col("precio_noche"))


val dominioCanalPorCiudad = dfConIngreso
  .groupBy("ciudad", "canal")
  .agg(
    sum("ingreso_reserva").alias("ingreso_total"),
    count("id_reserva").alias("num_reservas")
  )
  .orderBy(col("ciudad").asc, col("ingreso_total").desc)

// 5. Mostrar resultado
dominioCanalPorCiudad.show()


+---------+-------+-------------+------------+
|   ciudad|  canal|ingreso_total|num_reservas|
+---------+-------+-------------+------------+
|Barcelona|Booking|        360.0|           1|
|   Bilbao|Directo|        425.0|           2|
|   Bilbao| Airbnb|        195.0|           1|
|   Madrid| Airbnb|        665.0|           2|
|  Sevilla|Booking|        220.0|           1|
| Valencia|Directo|        360.0|           2|
| Valencia|Booking|        225.0|           1|
+---------+-------+-------------+------------+



import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.{datediff, to_date, col, sum, count}
spark: SparkSession = org.apache.spark.sql.SparkSession@3a466b35
datosReservas: Seq[(String, String, String, Double, String, String)] = List(
  ("R001", "2025-01-10", "2025-01-13", 95.0, "Airbnb", "Madrid"),
  ("R002", "2025-01-12", "2025-01-15", 120.0, "Booking", "Barcelona"),
  ("R003", "2025-01-20", "2025-01-22", 75.0, "Directo", "Valencia"),
  ("R004", "2025-02-01", "2025-02-05", 95.0, "Airbnb", "Madrid"),
  ("R005", "2025-02-10", "2025-02-12", 110.0, "Booking", "Sevilla"),
  ("R007", "2025-02-20", "2025-02-23", 85.0, "Directo", "Bilbao"),
  ("R008", "2025-03-01", "2025-03-04", 75.0, "Booking", "Valencia"),
  ("R013", "2025-04-08", "2025-04-10", 85.0, "Directo", "Bilbao"),
  ("R016", "2025-05-01", "2025-05-04", 70.0, "Directo", "Valencia"),
  ("R020", "2025-05-25", "2025-05-28", 65.0, "Airbnb", "Bilba

Pregunta 10 — ¿Qué apartamento tiene la mejor posición de ingreso dentro de su ciudad?
Lo que debes hacer:

Calcula primero ingreso_reserva si no existe todavía.
Agrupa por id_apartamento y ciudad, calcula ingreso_total = suma de ingreso_reserva.
Define una ventana ventanaCiudad con partitionBy("ciudad") y orderBy($"ingreso_total".desc).
Añade la columna ranking_ciudad usando rank() sobre esa ventana.
Filtra para ver solo el apartamento en posición 1 de cada ciudad.
Ordena por ingreso_total descendente.

In [12]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.{datediff, to_date, col, sum, rank}
import org.apache.spark.sql.expressions.Window

// 1. Iniciar sesión y apagar registros molestos
val spark = SparkSession.builder()
  .appName("Pregunta10")
  .master("local[*]")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

// 2. Cargar listado completo de datos
val datosReservas = Seq(
  ("R001","APT001","2025-01-10","2025-01-13",95.0,"Madrid"),
  ("R002","APT002","2025-01-12","2025-01-15",120.0,"Barcelona"),
  ("R003","APT003","2025-01-20","2025-01-22",75.0,"Valencia"),
  ("R004","APT001","2025-02-01","2025-02-05",95.0,"Madrid"),
  ("R005","APT004","2025-02-10","2025-02-12",110.0,"Sevilla"),
  ("R007","APT005","2025-02-20","2025-02-23",85.0,"Bilbao"),
  ("R008","APT003","2025-03-01","2025-03-04",75.0,"Valencia"),
  ("R013","APT005","2025-04-08","2025-04-10",85.0,"Bilbao"),
  ("R016","APT008","2025-05-01","2025-05-04",70.0,"Valencia"),
  ("R020","APT009","2025-05-25","2025-05-28",65.0,"Bilbao")
)

val schema = StructType(Array(
  StructField("id_reserva", StringType, true),
  StructField("id_apartamento", StringType, true),
  StructField("fecha_entrada", StringType, true),
  StructField("fecha_salida", StringType, true),
  StructField("precio_noche", DoubleType, true),
  StructField("ciudad", StringType, true)
))

val rdd = spark.sparkContext.parallelize(datosReservas).map(p => Row(p._1, p._2, p._3, p._4, p._5, p._6))
val reservas = spark.createDataFrame(rdd, schema)

// 3. Paso previo: Calcular ingreso por reserva
val dfConIngreso = reservas
  .withColumn("num_noches", datediff(to_date(col("fecha_salida"), "yyyy-MM-dd"), to_date(col("fecha_entrada"), "yyyy-MM-dd")))
  .withColumn("ingreso_reserva", col("num_noches") * col("precio_noche"))

// 4. Agrupar por apartamento y ciudad
val ingresosPorApto = dfConIngreso
  .groupBy("id_apartamento", "ciudad")
  .agg(sum("ingreso_reserva").alias("ingreso_total"))

// 5. Definir la ventana y aplicar el Ranking
val ventanaCiudad = Window.partitionBy("ciudad").orderBy(col("ingreso_total").desc)

val rankingApartamentos = ingresosPorApto
  .withColumn("ranking_ciudad", rank().over(ventanaCiudad))
  .filter(col("ranking_ciudad") === 1)
  .orderBy(col("ingreso_total").desc)

// 6. Mostrar el resultado final
rankingApartamentos.show()


+--------------+---------+-------------+--------------+
|id_apartamento|   ciudad|ingreso_total|ranking_ciudad|
+--------------+---------+-------------+--------------+
|        APT001|   Madrid|        665.0|             1|
|        APT005|   Bilbao|        425.0|             1|
|        APT003| Valencia|        375.0|             1|
|        APT002|Barcelona|        360.0|             1|
|        APT004|  Sevilla|        220.0|             1|
+--------------+---------+-------------+--------------+



import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.{datediff, to_date, col, sum, rank}
import org.apache.spark.sql.expressions.Window
spark: SparkSession = org.apache.spark.sql.SparkSession@3a466b35
datosReservas: Seq[(String, String, String, String, Double, String)] = List(
  ("R001", "APT001", "2025-01-10", "2025-01-13", 95.0, "Madrid"),
  ("R002", "APT002", "2025-01-12", "2025-01-15", 120.0, "Barcelona"),
  ("R003", "APT003", "2025-01-20", "2025-01-22", 75.0, "Valencia"),
  ("R004", "APT001", "2025-02-01", "2025-02-05", 95.0, "Madrid"),
  ("R005", "APT004", "2025-02-10", "2025-02-12", 110.0, "Sevilla"),
  ("R007", "APT005", "2025-02-20", "2025-02-23", 85.0, "Bilbao"),
  ("R008", "APT003", "2025-03-01", "2025-03-04", 75.0, "Valencia"),
  ("R013", "APT005", "2025-04-08", "2025-04-10", 85.0, "Bilbao"),
  ("R016", "APT008", "2025-05-01", "2025-05-04", 70.0, "Valencia"),
  ("R020", "APT009", "

Pregunta 19 — ¿Cómo podemos presentar las fechas de entrada en formato europeo?
El equipo de atención al cliente necesita las fechas en formato dd/MM/yyyy para los informes que envían a los propietarios.

Lo que debes hacer:

Sobre reservas, añade la columna fecha_entrada_es usando date_format($"fecha_entrada", "dd/MM/yyyy").
Añade también mes_entrada = month($"fecha_entrada") y anio_entrada = year($"fecha_entrada").
Selecciona id_reserva, ciudad, fecha_entrada, fecha_entrada_es, mes_entrada, anio_entrada.

In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.sum

val spark = SparkSession.builder()
  .appName("Ejer12")
  .master("local[*]")
  .getOrCreate()

// 2. Datos crudos
val datosPivot = Seq(
  ("Madrid", "Airbnb", 285.0),
  ("Barcelona", "Booking", 360.0),
  ("Valencia", "Directo", 150.0),
  ("Madrid", "Airbnb", 380.0),
  ("Bilbao", "Directo", 255.0),
  ("Valencia", "Booking", 225.0),
  ("Bilbao", "Directo", 170.0)
)


val schema = StructType(Array(
  StructField("ciudad", StringType, true),
  StructField("canal", StringType, true),
  StructField("ingreso_reserva", DoubleType, true)
))

val rdd = spark.sparkContext.parallelize(datosPivot).map(p => Row(p._1, p._2, p._3))
val resultadoFinal = spark.createDataFrame(rdd, schema)

val facturacionPivot = resultadoFinal
  .groupBy("ciudad")
  .pivot("canal")
  .agg(sum("ingreso_reserva"))
  .orderBy("ciudad")

facturacionPivot.show()


+---------+------+-------+-------+
|   ciudad|Airbnb|Booking|Directo|
+---------+------+-------+-------+
|Barcelona|  NULL|  360.0|   NULL|
|   Bilbao|  NULL|   NULL|  425.0|
|   Madrid| 665.0|   NULL|   NULL|
| Valencia|  NULL|  225.0|  150.0|
+---------+------+-------+-------+



import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Row
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions.sum
spark: SparkSession = org.apache.spark.sql.SparkSession@3a466b35
datosPivot: Seq[(String, String, Double)] = List(
  ("Madrid", "Airbnb", 285.0),
  ("Barcelona", "Booking", 360.0),
  ("Valencia", "Directo", 150.0),
  ("Madrid", "Airbnb", 380.0),
  ("Bilbao", "Directo", 255.0),
  ("Valencia", "Booking", 225.0),
  ("Bilbao", "Directo", 170.0)
)
schema: StructType = Seq(
  StructField(
    name = "ciudad",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "canal",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "ingreso_reserva",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  )
)
rdd: org.apache.spark.rdd.RDD[Row] = MapPartitionsRDD[1] at map at cmd6.sc:30
resultadoFinal: org.apache.spark.sql.package.DataFrame = [ciudad: stri